# Post-Race Debrief

Turn the replays into a lap-by-lap debrief table, read what Knowledge Catalog knows about it (data products, quality scores, lineage) and ask the data agent about the race.

This notebook follows the same steps as `docs/02-debrief.md`. Run the cells in order. Do `01 Race Strategy` first, or have one finished replay. The profile, insights, glossary and aspects are easiest to read in the console: Step 4 of the guide shows where.

In [ ]:
import google.auth
import pandas as pd
from google.auth.transport.requests import AuthorizedSession
from google.cloud import bigquery

# The region and BigQuery location from infra/variables.tf (var.region and
# var.dataset_location). Change them here if you changed them there. Knowledge
# Catalog keeps BigQuery metadata in the dataset's location, in lower case.
REGION = "us-central1"
BQ_LOCATION = "US"
CATALOG_LOCATION = BQ_LOCATION.lower()

credentials, PROJECT_ID = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
session = AuthorizedSession(credentials)
bq = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)


def run_sql(sql):
    """Run a query from this notebook, with the project and region filled in, and return a DataFrame."""
    sql = sql.replace("${project_id}", PROJECT_ID).replace("${region}", REGION)
    return bq.query(sql).to_dataframe()


print("Project:", PROJECT_ID)

## Step 1: Build the debrief

Rebuild `serving.lap_debrief` from every replay with one batch query: one row per car per lap, with the JSON payloads turned into typed columns. Then show the latest replay's top five. Saved query **07 Debrief: every car, every lap**.

In [ ]:
DEBRIEF = r"""
-- 07 Debrief: every car, every lap
-- The post-race debrief table. One row per car per completed lap, for every
-- replay, with the JSON payloads in raw.raw_events turned into typed columns:
-- position, gaps, tyres, pit stops and the weather when the lap ended, next to
-- the moments the continuous queries flagged and the calls Gemini made.
--
-- The data agent (infra/agent.tf) answers questions from this table, and
-- Knowledge Catalog scans it (infra/data_scans.tf). The apply runs it once.
-- Run it again after a replay to add that replay: it rebuilds the whole table
-- with one batch query, and BigQuery records the lineage from the three source
-- tables each time.
CREATE OR REPLACE TABLE `${project_id}.serving.lap_debrief`
CLUSTER BY run_id, driver_number
OPTIONS (
  description = 'One row per car per completed lap, for every replay. Written by the saved query 07 Debrief.',
  -- The Insights tab in BigQuery shows the Knowledge Catalog insights scan
  -- named here (infra/data_scans.tf). CREATE OR REPLACE drops labels, so they
  -- are set on every run.
  labels = [
    ('dataplex-data-documentation-published-scan', 'lap-debrief-insights'),
    ('dataplex-data-documentation-published-project', '${project_id}'),
    ('dataplex-data-documentation-published-location', '${region}')
  ]
)
AS
WITH ev AS (
  SELECT run_id, event_type, event_ts, publish_time, driver_number, payload
  FROM `${project_id}.raw.raw_events`
  WHERE event_type IN ('drivers', 'laps', 'position', 'intervals', 'stints', 'pit', 'weather')
),

-- When each replay ran. Race times repeat across replays, publish times do not.
replays AS (
  SELECT run_id, MIN(publish_time) AS replayed_at
  FROM ev WHERE event_type = 'drivers'
  GROUP BY run_id
),

drivers AS (
  SELECT
    run_id,
    driver_number,
    ANY_VALUE(JSON_VALUE(payload, '$.name_acronym')) AS driver,
    ANY_VALUE(JSON_VALUE(payload, '$.team_name')) AS team
  FROM ev WHERE event_type = 'drivers'
  GROUP BY run_id, driver_number
),

-- A lap is published when it ends, so its event_ts is the end of the lap.
-- Pub/Sub can deliver a message twice: keep the first copy.
laps AS (
  SELECT
    run_id,
    driver_number,
    SAFE.INT64(payload.lap_number) AS lap,
    event_ts AS lap_end_ts,
    SAFE.FLOAT64(payload.lap_duration) AS lap_s,
    SAFE.BOOL(payload.is_pit_out_lap) AS pit_out_lap
  FROM ev WHERE event_type = 'laps'
  QUALIFY ROW_NUMBER() OVER (PARTITION BY run_id, driver_number, lap ORDER BY publish_time) = 1
),

-- The latest reading at or before the end of each lap.
positions AS (
  SELECT l.run_id, l.driver_number, l.lap, SAFE.INT64(e.payload.position) AS position
  FROM laps AS l
  JOIN ev AS e
    ON e.event_type = 'position' AND e.run_id = l.run_id
    AND e.driver_number = l.driver_number AND e.event_ts <= l.lap_end_ts
  QUALIFY ROW_NUMBER() OVER (PARTITION BY l.run_id, l.driver_number, l.lap ORDER BY e.event_ts DESC) = 1
),

-- gap_to_leader and interval are numbers, or text such as "+1 LAP".
gaps AS (
  SELECT
    l.run_id, l.driver_number, l.lap,
    SAFE.FLOAT64(e.payload.gap_to_leader) AS gap_to_leader_s,
    SAFE.FLOAT64(e.payload.`interval`) AS interval_s
  FROM laps AS l
  JOIN ev AS e
    ON e.event_type = 'intervals' AND e.run_id = l.run_id
    AND e.driver_number = l.driver_number AND e.event_ts <= l.lap_end_ts
  QUALIFY ROW_NUMBER() OVER (PARTITION BY l.run_id, l.driver_number, l.lap ORDER BY e.event_ts DESC) = 1
),

-- The stint a lap belongs to. A stint is published again with lap_end once it
-- ends, so take the latest stint that started on or before the lap.
tyres AS (
  SELECT
    l.run_id, l.driver_number, l.lap,
    JSON_VALUE(e.payload, '$.compound') AS compound,
    l.lap - SAFE.INT64(e.payload.lap_start) + SAFE.INT64(e.payload.tyre_age_at_start) AS tyre_age_laps
  FROM laps AS l
  JOIN ev AS e
    ON e.event_type = 'stints' AND e.run_id = l.run_id
    AND e.driver_number = l.driver_number AND SAFE.INT64(e.payload.lap_start) <= l.lap
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY l.run_id, l.driver_number, l.lap
    ORDER BY SAFE.INT64(e.payload.stint_number) DESC, e.event_ts DESC) = 1
),

pits AS (
  SELECT
    run_id,
    driver_number,
    SAFE.INT64(payload.lap_number) AS lap,
    SAFE.FLOAT64(payload.lane_duration) AS pit_lane_s,
    SAFE.FLOAT64(payload.stop_duration) AS stop_s
  FROM ev WHERE event_type = 'pit'
  QUALIFY ROW_NUMBER() OVER (PARTITION BY run_id, driver_number, lap ORDER BY publish_time) = 1
),

weather AS (
  SELECT l.run_id, l.driver_number, l.lap, SAFE.INT64(e.payload.rainfall) = 1 AS raining
  FROM laps AS l
  JOIN ev AS e
    ON e.event_type = 'weather' AND e.run_id = l.run_id AND e.event_ts <= l.lap_end_ts
  QUALIFY ROW_NUMBER() OVER (PARTITION BY l.run_id, l.driver_number, l.lap ORDER BY e.event_ts DESC) = 1
),

-- A GAP_CLOSING moment has no lap, so it goes on the first lap the car
-- finished at or after the moment. Calls answer one moment each, through the
-- moment's Pub/Sub message ID. A moment can be flagged twice after a
-- continuous query restarts: list it once.
car_moments AS (
  SELECT m.run_id, m.driver_number, IFNULL(m.lap, l.lap) AS lap, m.moment, m.message_id
  FROM `${project_id}.serving.moments` AS m
  LEFT JOIN laps AS l
    ON m.lap IS NULL AND l.run_id = m.run_id
    AND l.driver_number = m.driver_number AND l.lap_end_ts >= m.event_ts
  WHERE m.driver_number IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY m.run_id, m.driver_number, m.moment, m.event_ts ORDER BY l.lap_end_ts) = 1
),

moments AS (
  SELECT
    m.run_id,
    m.driver_number,
    m.lap,
    ARRAY_AGG(DISTINCT m.moment ORDER BY m.moment) AS moments,
    ARRAY_AGG(c.call_text IGNORE NULLS ORDER BY c.created_at LIMIT 1)[SAFE_OFFSET(0)] AS call_text
  FROM car_moments AS m
  LEFT JOIN `${project_id}.serving.calls` AS c
    ON c.moment_message_id = m.message_id AND c.status = 'OK'
  WHERE m.lap IS NOT NULL
  GROUP BY m.run_id, m.driver_number, m.lap
)

SELECT
  l.run_id,
  r.replayed_at,
  l.lap,
  l.driver_number,
  d.driver,
  d.team,
  p.position,
  l.lap_s,
  g.gap_to_leader_s,
  g.interval_s,
  t.compound,
  t.tyre_age_laps,
  pt.lap IS NOT NULL AS pitted,
  pt.pit_lane_s,
  pt.stop_s,
  IFNULL(l.pit_out_lap, FALSE) AS pit_out_lap,
  IFNULL(w.raining, FALSE) AS raining,
  IFNULL(m.moments, []) AS moments,
  m.call_text,
  l.lap_end_ts
FROM laps AS l
JOIN replays AS r USING (run_id)
LEFT JOIN drivers AS d USING (run_id, driver_number)
LEFT JOIN positions AS p USING (run_id, driver_number, lap)
LEFT JOIN gaps AS g USING (run_id, driver_number, lap)
LEFT JOIN tyres AS t USING (run_id, driver_number, lap)
LEFT JOIN pits AS pt USING (run_id, driver_number, lap)
LEFT JOIN weather AS w USING (run_id, driver_number, lap)
LEFT JOIN moments AS m USING (run_id, driver_number, lap)
WHERE l.lap IS NOT NULL;

-- The latest replay, lap by lap for the top five.
SELECT lap, position, driver, team, lap_s, compound, tyre_age_laps, pitted, raining, moments
FROM `${project_id}.serving.lap_debrief`
WHERE run_id = (SELECT run_id FROM `${project_id}.serving.lap_debrief` ORDER BY replayed_at DESC LIMIT 1)
  AND position <= 5
ORDER BY lap DESC, position
LIMIT 25;
"""
run_sql(DEBRIEF)

## Step 2: Find the data products

The two data products and the tables in each.

In [ ]:
base = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}/dataProducts"
for product in session.get(base).json().get("dataProducts", []):
    print(product["displayName"], "|", product.get("ownerEmails", []))
    print("  ", product.get("description", ""))
    assets = session.get(f"https://dataplex.googleapis.com/v1/{product['name']}/dataAssets").json()
    for asset in assets.get("dataAssets", []):
        print("   -", asset["resource"].split("/datasets/")[-1].replace("/tables/", "."))

## Step 3: Read the quality scores

The latest result of each quality scan, rule by rule. Run the scans again after a replay from the table's **Data quality** tab, or with `gcloud dataplex datascans run`.

In [ ]:
rows = []
for scan in ["raw-events-quality", "moments-quality", "calls-quality", "lap-debrief-quality"]:
    name = f"projects/{PROJECT_ID}/locations/{REGION}/dataScans/{scan}"
    jobs = session.get(
        f"https://dataplex.googleapis.com/v1/{name}/jobs", params={"pageSize": 1}
    ).json()
    if not jobs.get("dataScanJobs"):
        print(scan, "has not run yet")
        continue
    job = session.get(
        f"https://dataplex.googleapis.com/v1/{jobs['dataScanJobs'][0]['name']}",
        params={"view": "FULL"},
    ).json()
    for rule in job.get("dataQualityResult", {}).get("rules", []):
        rows.append(
            {
                "scan": scan,
                "rule": rule["rule"]["name"],
                "dimension": rule["rule"]["dimension"],
                "passed": rule.get("passed", False),
                "pass_ratio": rule.get("passRatio"),
                "failing_rows": rule.get("assertionRowCount"),
            }
        )
pd.DataFrame(rows)

## Step 5: Follow the lineage

The tables that feed `serving.lap_debrief`, as recorded by the Data Lineage API from the BigQuery jobs that wrote it. A new job can take a few minutes to appear.

In [ ]:
links = session.post(
    f"https://datalineage.googleapis.com/v1/projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}:searchLinks",
    json={"target": {"fullyQualifiedName": f"bigquery:{PROJECT_ID}.serving.lap_debrief"}},
).json()
for link in links.get("links", []):
    print(link["source"]["fullyQualifiedName"], "->", link["target"]["fullyQualifiedName"])

## Step 6: Ask the data agent

Ask **Pit wall race engineer** a question through the Conversational Analytics API, and print the SQL it ran and its answer. The agent is easier to use under **BigQuery > Agents**.

In [ ]:
QUESTION = "Did the drivers Gemini told to box when the rain started actually pit within the next two laps?"

agents = session.get(
    f"https://geminidataanalytics.googleapis.com/v1beta/projects/{PROJECT_ID}/locations/global/dataAgents"
).json()
agent = next(
    a["name"]
    for a in agents.get("dataAgents", [])
    if a.get("displayName") == "Pit wall race engineer"
)

reply = session.post(
    f"https://geminidataanalytics.googleapis.com/v1beta/projects/{PROJECT_ID}/locations/global:chat",
    json={
        "parent": f"projects/{PROJECT_ID}/locations/global",
        "messages": [{"userMessage": {"text": QUESTION}}],
        "dataAgentContext": {"dataAgent": agent},
    },
    timeout=300,
).json()

sql, answer = None, []
for message in reply:
    system = message.get("systemMessage", {})
    sql = system.get("data", {}).get("generatedSql", sql)
    if system.get("text", {}).get("textType") == "FINAL_RESPONSE":
        answer += system["text"]["parts"]
print(sql)
print()
print("\n".join(answer))